In [5]:
%%writefile worker.py
import time
import pandas as pd
import traceback
import os
import json
from datetime import datetime
from easynmt import EasyNMT
import nltk
import re

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

def split_into_sentences(text, source_lang='en', max_sentence_length=500):
    """
    Split text into sentences with language-specific handling

    Args:
        text: Input text
        source_lang: Source language code
        max_sentence_length: Maximum characters per sentence (for fallback splitting)
    """

    # SPECIAL CASE: Czech uses double spaces as sentence delimiters
    if source_lang == 'cs':
        # Split on double (or more) spaces
        sentences = re.split(r'\s{2,}', text)
        sentences = [s.strip() for s in sentences if s.strip() and len(s.strip()) > 5]

        if sentences:
            return sentences

    # For other languages, try NLTK first
    try:
        sentences = nltk.sent_tokenize(text)

        # Check if NLTK failed (returns only 1 sentence for long text)
        if len(sentences) == 1 and len(text) > 1000:
            # Fallback: split on common sentence endings + newlines
            sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
            sentences = [s.strip() for s in sentences if s.strip()]

        # If still just one long sentence, force split by length
        if len(sentences) == 1 and len(text) > max_sentence_length:
            # Split on any reasonable boundary (punctuation or newline)
            sentences = re.split(r'[.!?;:\n]+', text)
            sentences = [s.strip() for s in sentences if s.strip() and len(s) > 10]

        # If STILL one sentence, split by character limit as last resort
        if len(sentences) == 1 and len(text) > max_sentence_length:
            # Split into chunks of max_sentence_length at word boundaries
            words = text.split()
            sentences = []
            current = []
            current_len = 0

            for word in words:
                if current_len + len(word) + 1 > max_sentence_length and current:
                    sentences.append(' '.join(current))
                    current = [word]
                    current_len = len(word)
                else:
                    current.append(word)
                    current_len += len(word) + 1

            if current:
                sentences.append(' '.join(current))

        return sentences

    except Exception as e:
        # Ultimate fallback: return as-is
        print(f"Warning: sentence splitting failed: {e}", flush=True)
        return [text]

def process_single_file(csv_file, data_dir, source_lang, target_lang='en', year_cutoff=None, batch_size=32):
    """
    Process using EXACT ParlaMint methodology:
    - OPUS-MT models via EasyNMT
    - Sentence-by-sentence translation
    - Language-specific sentence splitting
    """

    try:
        name = os.path.splitext(os.path.basename(csv_file))[0]
        checkpoint_file = f"checkpoint_{name}.json"
        output_file = f"translated_{name}.csv"
        stats_file = f"translation_stats_{name}.json"

        if os.path.exists(output_file):
            print(f"[{csv_file}] ✅ Already completed, skipping.", flush=True)
            return f"✅ {csv_file}: Already completed (found {output_file})"

        start_idx = 0
        sentence_stats = {'total_texts': 0, 'total_sentences': 0, 'avg_sentences_per_text': 0, 'single_sentence_texts': 0}

        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'r') as f:
                checkpoint = json.load(f)
                start_idx = checkpoint['last_completed_idx']
                if 'sentence_stats' in checkpoint:
                    sentence_stats = checkpoint['sentence_stats']
                print(f"[{csv_file}] 📂 Resuming from checkpoint at index {start_idx}", flush=True)

        print(f"[{csv_file}] 🔧 Loading OPUS-MT model via EasyNMT (lang: {source_lang})...", flush=True)

        # This is EXACTLY what ParlaMint used
        model = EasyNMT('opus-mt')

        print(f"[{csv_file}] 📖 Reading CSV...", flush=True)
        path = os.path.join(data_dir, csv_file)
        df = pd.read_csv(path)

        if "text" not in df.columns:
            return f"⚠️ {csv_file}: No 'text' column"

        original_count = len(df)
        if year_cutoff is not None and "date" in df.columns:
            df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
            df = df[df['year'] < year_cutoff].copy()
            df = df.drop('year', axis=1)
            print(f"[{csv_file}] 📅 Filtered: {len(df)}/{original_count} rows with year < {year_cutoff}", flush=True)

        if len(df) == 0:
            return f"⚠️ {csv_file}: No rows to translate after filtering"

        texts_to_translate = df["text"].dropna().astype(str).tolist()
        valid_indices = df["text"].dropna().index.tolist()

        if not texts_to_translate:
            return f"⚠️ {csv_file}: No valid text"

        total_texts = len(texts_to_translate)

        # Special message for Czech
        if source_lang == 'cs':
            print(f"[{csv_file}] 🇨🇿 Czech detected: using double-space sentence splitting", flush=True)

        print(f"[{csv_file}] 🚀 Processing {total_texts} texts (sentence-by-sentence like ParlaMint)...", flush=True)

        if start_idx == 0:
            df['en_translation'] = None
        else:
            if os.path.exists(f"partial_{output_file}"):
                partial_df = pd.read_csv(f"partial_{output_file}")
                df = partial_df

        file_start_time = time.time()
        last_checkpoint_time = time.time()

        for i in range(start_idx, total_texts, batch_size):
            batch_start = time.time()

            batch_texts = texts_to_translate[i:i+batch_size]
            batch_indices = valid_indices[i:i+batch_size]

            # EXPLICIT sentence-level translation (ParlaMint approach)
            batch_translated = []

            for text in batch_texts:
                # Split into sentences with language-specific handling
                sentences = split_into_sentences(text, source_lang=source_lang, max_sentence_length=500)
                sentence_stats['total_sentences'] += len(sentences)

                # Track texts that only have 1 sentence (potential issue)
                if len(sentences) == 1 and len(text) > 500:
                    sentence_stats['single_sentence_texts'] += 1

                # Translate each sentence individually
                if len(sentences) > 0:
                    translated_sentences = model.translate(
                        sentences,
                        source_lang=source_lang,
                        target_lang=target_lang
                    )
                    # Recombine into full text
                    full_translation = ' '.join(translated_sentences)
                    batch_translated.append(full_translation)
                else:
                    batch_translated.append('')

                sentence_stats['total_texts'] += 1

            # Update statistics
            if sentence_stats['total_texts'] > 0:
                sentence_stats['avg_sentences_per_text'] = sentence_stats['total_sentences'] / sentence_stats['total_texts']

            for j, translation in enumerate(batch_translated):
                df_idx = batch_indices[j]
                df.at[df_idx, 'en_translation'] = translation

            batch_duration = time.time() - batch_start
            batch_num = (i - start_idx) // batch_size + 1

            if batch_num % 10 == 0 or (time.time() - last_checkpoint_time) > 300:
                elapsed = time.time() - file_start_time
                progress = (i + len(batch_texts)) / total_texts * 100
                speed = (i + len(batch_texts) - start_idx) / elapsed
                remaining = (total_texts - i - len(batch_texts)) / speed if speed > 0 else 0

                avg_sent = sentence_stats['avg_sentences_per_text']
                single_sent_pct = (sentence_stats['single_sentence_texts'] / sentence_stats['total_texts'] * 100) if sentence_stats['total_texts'] > 0 else 0

                print(f"[{csv_file}] {progress:.1f}% | {i+len(batch_texts)}/{total_texts} | "
                      f"{speed:.1f} texts/sec | Avg {avg_sent:.1f} sent/text | "
                      f"Single-sent: {single_sent_pct:.1f}% | ETA: {remaining/60:.1f} min", flush=True)

                checkpoint = {
                    'last_completed_idx': i + len(batch_texts),
                    'timestamp': datetime.now().isoformat(),
                    'progress_pct': progress,
                    'speed': speed,
                    'sentence_stats': sentence_stats
                }
                with open(checkpoint_file, 'w') as f:
                    json.dump(checkpoint, f)

                df.to_csv(f"partial_{output_file}", index=False)
                last_checkpoint_time = time.time()

        total_duration = time.time() - file_start_time
        speed = total_texts / total_duration

        df.to_csv(output_file, index=False)

        single_sent_pct = (sentence_stats['single_sentence_texts'] / sentence_stats['total_texts'] * 100) if sentence_stats['total_texts'] > 0 else 0

        final_stats = {
            'model': 'OPUS-MT via EasyNMT (ParlaMint sentence-by-sentence methodology)',
            'source_language': source_lang,
            'total_texts': sentence_stats['total_texts'],
            'total_sentences_translated': sentence_stats['total_sentences'],
            'avg_sentences_per_text': sentence_stats['avg_sentences_per_text'],
            'single_sentence_texts_pct': single_sent_pct,
            'translation_speed_texts_per_sec': speed,
            'total_duration_minutes': total_duration / 60
        }

        with open(stats_file, 'w') as f:
            json.dump(final_stats, f, indent=2)

        if os.path.exists(checkpoint_file):
            os.remove(checkpoint_file)
        if os.path.exists(f"partial_{output_file}"):
            os.remove(f"partial_{output_file}")

        result = (f"✅ {csv_file}: {total_texts} texts ({sentence_stats['total_sentences']} sentences, "
                 f"{single_sent_pct:.1f}% single-sent) in {total_duration/60:.1f} min ({speed:.1f} texts/sec)")
        print(result, flush=True)
        return result

    except Exception as e:
        error_details = f"❌ {csv_file} CRASHED: {type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        print(error_details, flush=True)

        if 'i' in locals():
            checkpoint = {
                'last_completed_idx': i,
                'timestamp': datetime.now().isoformat(),
                'error': str(e),
                'sentence_stats': sentence_stats if 'sentence_stats' in locals() else {}
            }
            with open(checkpoint_file, 'w') as f:
                json.dump(checkpoint, f)
            print(f"[{csv_file}] 💾 Checkpoint saved at index {i}", flush=True)

        return error_details

Overwriting worker.py


In [7]:
%%writefile run_parallel.py
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import os
import time
from datetime import datetime
from worker import process_single_file

# Language mapping - OPUS-MT language codes
# These match the ParlaMint methodology
lang_map = {
    "Bundestag": "de",       # German      # English (UK) - will skip translation
    "Congreso": "es",        # Spanish
    "Folketing": "da",       # Danish      # English (New Zealand) - will skip translation
    "PSP": "cs",             # Czech
    "TweedeKamer": "nl",     # Dutch
    "Corp_Riksdagen_V2": "sv"
}

# Year cutoff per country (None = process all, otherwise only < year)
year_cutoffs = {
    "Bundestag": None,       # Translate all
    "Congreso": None,        # Translate all
    "Folketing": None,       # Translate all
    "PSP": None,             # Translate all
    "TweedeKamer": None,     # Translate all
    "Corp_Riksdagen_V2": None
}

if __name__ == "__main__":
    mp.set_start_method('spawn', force=True)

    data_dir = "/home/tom/data/parlspeech"
    csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

    tasks = []
    for csv_file in csv_files:
        name = os.path.splitext(os.path.basename(csv_file))[0]
        source_lang = lang_map.get(name)

        if source_lang:
            # Skip files that are already in English
            if source_lang == 'en':
                print(f"⏭️  Skipping {csv_file} (already in English)")
                continue

            year_cutoff = year_cutoffs.get(name)
            target_lang = "en"  # All translate to English
            tasks.append((csv_file, data_dir, source_lang, target_lang, year_cutoff))

    print(f"\n{'='*80}")
    print(f"🚀 Starting translation using ParlaMint methodology")
    print(f"📚 Method: OPUS-MT via EasyNMT")
    print(f"📅 Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"📁 Files to process: {len(tasks)}")
    print(f"⚙️  Workers: 1")
    print(f"💾 Batch size: 32")
    print(f"\n📝 Reference: ParlaMint corpus translation methodology")
    print(f"   https://github.com/TajaKuzman/Parlamint-translation")
    print(f"\n📅 Year cutoffs:")
    for name, cutoff in year_cutoffs.items():
        if name not in ["Commons", "NZHoR"]:  # Skip English corpora
            if cutoff:
                print(f"   {name}: < {cutoff}")
            else:
                print(f"   {name}: All years")
    print(f"{'='*80}\n")

    overall_start = time.time()
    completed = 0
    failed = 0

    # Use 1 worker - OPUS-MT models are relatively lightweight
    # but EasyNMT downloads models on-demand which can cause issues with parallel processing
    with ProcessPoolExecutor(max_workers=1) as executor:
        futures = {
            executor.submit(process_single_file, csv_file, data_dir, source_lang,
                          target_lang, year_cutoff=year_cutoff, batch_size=32): csv_file
            for csv_file, data_dir, source_lang, target_lang, year_cutoff in tasks
        }

        for future in as_completed(futures):
            csv_file = futures[future]
            try:
                result = future.result(timeout=36000)  # 10 hour timeout per file
                print(f"\n{result}\n", flush=True)
                completed += 1
            except Exception as e:
                print(f"\n❌ {csv_file} failed: {type(e).__name__}: {e}\n", flush=True)
                failed += 1

            print(f"📊 Progress: {completed + failed}/{len(tasks)} files processed ({completed} ✅, {failed} ❌)", flush=True)

    overall_duration = time.time() - overall_start

    print(f"\n{'='*80}")
    print(f"✅ ALL PROCESSING COMPLETE")
    print(f"📅 End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"⏱️  Total time: {overall_duration/3600:.2f} hours")
    print(f"📊 Completed: {completed}/{len(tasks)}")
    print(f"❌ Failed: {failed}/{len(tasks)}")
    print(f"\n📝 Methodology: Replicated ParlaMint corpus translation approach")
    print(f"   Model: OPUS-MT (Helsinki-NLP)")
    print(f"   Framework: EasyNMT")
    print(f"   Translation level: Sentence-by-sentence with automatic chunking")
    print(f"{'='*80}\n")

Overwriting run_parallel.py


In [40]:

import torch
import gc

# Clear Python garbage
gc.collect()

# Clear CUDA cache
torch.cuda.empty_cache()

# Reset CUDA memory stats
torch.cuda.reset_peak_memory_stats()

print("✅ GPU memory cleared")

✅ GPU memory cleared


In [3]:
import pandas as pd

df = pd.read_csv("/home/tom/projects/corpora/translated_PSP.csv")
df_trans = df[df['en_translation'].notna()].copy()

print(f"Total rows: {len(df)}")
print(f"Translated rows: {len(df_trans)}")
print()

# Check a few long texts
long_texts = df_trans[df_trans['text'].str.len() > 2000].sample(n=min(3, len(df_trans)))

for idx, row in long_texts.iterrows():
    orig_text = row['text']
    translation = row['en_translation']

    print("="*80)
    print(f"Row {idx}")
    print("="*80)
    print(f"Original: {len(orig_text)} chars")
    print(f"Translation: {len(translation)} chars")
    print(f"Ratio: {len(translation)/len(orig_text):.2f}")
    print(f"\nOriginal (first 300): {orig_text[:300]}...")
    print(f"\nTranslation (first 300): {translation[:300]}...")
    print()

# Also check some short texts to see if quality is good
print("\n" + "="*80)
print("CHECKING SHORT TEXTS (< 500 chars)")
print("="*80)

short_texts = df_trans[df_trans['text'].str.len() < 500].sample(n=min(3, len(df_trans)))

for idx, row in short_texts.iterrows():
    orig_text = row['text']
    translation = row['en_translation']

    print(f"\nRow {idx}:")
    print(f"Original: {orig_text[:200]}")
    print(f"Translation: {translation[:200]}")
    print(f"Ratio: {len(translation)/len(orig_text):.2f}")

Total rows: 329135
Translated rows: 329135

Row 76724
Original: 4024 chars
Translation: 4335 chars
Ratio: 1.08

Original (first 300): Poslanec Jiří Václavek  Rozumím tomu  pane předsedající   Vážený pane předsedající  pane premiére  dámy a pánové  v rozpočtovém výboru byly projednávány dva návrhy na doporučení Poslanecké sněmovně  jak vám to před chvílí sdělil pan předseda kolega Tlustý  s tím  že jeho návrh neprošel a už jste o t...

Translation (first 300): Member Jiří Václavek I understand. Mr President Mr President Prime Minister Ladies and gentlemen two proposals for a recommendation to the Chamber of Deputies were discussed in the Committee on Budgets. as Mr. Chairman Fat told you a moment ago that his proposal failed, and you've heard of it, too. ...

Row 66843
Original: 9770 chars
Translation: 10187 chars
Ratio: 1.04

Original (first 300): Předseda vlády ČR Miloš Zeman  Děkuji  Vážený pane předsedající  vážené kolegyně poslankyně  vážení kolegové poslanci  rád bych navázal na ú

In [3]:
import pandas as pd
from transformers import NllbTokenizer

tokenizer = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")

df = pd.read_csv("/home/tom/projects/corpora/partial_translated_Bundestag.csv")

# Check row 382 (the long one)
row_382 = df.loc[382]

orig_text = row_382['text']
translation = row_382['en_translation']

print("="*80)
print("ROW 382 ANALYSIS")
print("="*80)

orig_tokens = tokenizer(str(orig_text), return_tensors="pt")
trans_tokens = tokenizer(str(translation), return_tensors="pt")

print(f"Original: {len(orig_text)} chars, {orig_tokens['input_ids'].shape[1]} tokens")
print(f"Translation: {len(translation)} chars, {trans_tokens['input_ids'].shape[1]} tokens")
print(f"Ratio: {len(translation)/len(orig_text):.2f}")

if orig_tokens['input_ids'].shape[1] > 900:
    print(f"\n✅ This WAS chunked (original had {orig_tokens['input_ids'].shape[1]} tokens)")
else:
    print(f"\n❌ This should NOT have been chunked")

/tmp/ipykernel_51779/164075701.py:6: DtypeWarning: Columns (2,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/tom/projects/corpora/partial_translated_Bundestag.csv")


In [14]:
df = pd.read_csv("/home/tom/projects/corpora/translated_Bundestag.csv")
df.head()

,Unnamed: 0,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country,en_translation
0,2,1991-03-12,NaN,2,Theodor Waigel,CDU/CSU,211.0,False,880,Frau Präsidentin ! Meine sehr geehrten Damen u...,DE-Bundestag,DEU,Madam President! Ladies and gentlemen! The dra...
1,3,1991-03-12,NaN,3,Rita Süssmuth,CDU/CSU,211.0,True,15,"Herr Duve , ich fordere Sie auf , Herrn Waigel...",DE-Bundestag,DEU,"Mr Duve , I ask you to let Mr Waigel speak . ."
2,4,1991-03-12,NaN,4,Theodor Waigel,CDU/CSU,211.0,False,5756,Nicht viel besser sieht es im Straßenwesen aus...,DE-Bundestag,DEU,It doesn't look much better in the street worl...
3,5,1991-03-12,NaN,5,Rita Süssmuth,CDU/CSU,211.0,True,69,"Meine Damen und Herren , wie bereits heute mor...",DE-Bundestag,DEU,"Ladies and gentlemen , as already announced th..."
4,6,1991-03-12,NaN,6,Barbara Höll,PDS/LINKE,86.0,False,233,Geehrte Frau Präsidentin ! Meine Damen und Her...,DE-Bundestag,DEU,Madam President! Ladies and gentlemen! The pos...


In [6]:
import pandas as pd
import re
from collections import Counter

# Load Czech dataset
df = pd.read_csv("/home/tom/data/parlspeech/PSP.csv")

print("="*80)
print("CZECH DATASET (PSP) INSPECTION")
print("="*80)

print(f"\nTotal rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Check text column
if 'text' in df.columns:
    texts = df['text'].dropna().astype(str)
    print(f"\nRows with text: {len(texts)}")

    # Sample a few texts
    print("\n" + "="*80)
    print("SAMPLE TEXTS (first 3)")
    print("="*80)
    for i, text in enumerate(texts.head(3).tolist()):
        print(f"\nSample {i+1} (length: {len(text)} chars):")
        print(text[:500])
        print("..." if len(text) > 500 else "")

    # Analyze punctuation
    print("\n" + "="*80)
    print("PUNCTUATION ANALYSIS")
    print("="*80)

    all_text = ' '.join(texts.head(100).tolist())  # Sample first 100 texts

    # Count common punctuation marks
    punctuation_counts = {
        'Period (.)': all_text.count('.'),
        'Exclamation (!)': all_text.count('!'),
        'Question (?)': all_text.count('?'),
        'Comma (,)': all_text.count(','),
        'Semicolon (;)': all_text.count(';'),
        'Colon (:)': all_text.count(':'),
        'Newline (\\n)': all_text.count('\n'),
        'Tab (\\t)': all_text.count('\t'),
    }

    print("\nPunctuation marks in first 100 texts:")
    for mark, count in punctuation_counts.items():
        print(f"  {mark}: {count}")

    # Check for unusual patterns
    print("\n" + "="*80)
    print("SENTENCE STRUCTURE ANALYSIS")
    print("="*80)

    # Try splitting by periods
    sample_text = texts.iloc[0]
    print(f"\nSample text length: {len(sample_text)} chars")

    # Try different splitting methods
    nltk_split = re.split(r'(?<=[.!?])\s+', sample_text)
    print(f"Split by [.!?] + space: {len(nltk_split)} segments")

    newline_split = sample_text.split('\n')
    print(f"Split by newlines: {len(newline_split)} segments")

    # Check if sentences are separated by double spaces or other patterns
    double_space_split = sample_text.split('  ')
    print(f"Split by double spaces: {len(double_space_split)} segments")

    # Look for common Czech sentence patterns
    print("\n" + "="*80)
    print("LOOKING FOR ALTERNATIVE DELIMITERS")
    print("="*80)

    # Check first text in detail
    print(f"\nFirst 1000 chars with visible whitespace:")
    print(repr(sample_text[:1000]))

    # Check character distribution
    char_counter = Counter(sample_text)
    print(f"\nMost common characters (top 20):")
    for char, count in char_counter.most_common(20):
        if char == '\n':
            print(f"  '\\n' (newline): {count}")
        elif char == '\t':
            print(f"  '\\t' (tab): {count}")
        elif char == ' ':
            print(f"  ' ' (space): {count}")
        else:
            print(f"  '{char}': {count}")

    # Check average text length
    print("\n" + "="*80)
    print("TEXT LENGTH STATISTICS")
    print("="*80)
    print(texts.str.len().describe())

    # Check for very long texts without periods
    long_texts = texts[texts.str.len() > 1000]
    print(f"\nTexts > 1000 chars: {len(long_texts)}")

    if len(long_texts) > 0:
        sample_long = long_texts.iloc[0]
        periods_in_long = sample_long.count('.')
        print(f"Sample long text: {len(sample_long)} chars, {periods_in_long} periods")
        print(f"First 500 chars:")
        print(sample_long[:500])

else:
    print("⚠️ No 'text' column found!")

CZECH DATASET (PSP) INSPECTION

Total rows: 329135
Columns: ['Unnamed: 0', 'date', 'agenda', 'speechnumber', 'speaker', 'party', 'party.facts.id', 'chair', 'terms', 'text', 'parliament', 'iso3country']

Rows with text: 329135

SAMPLE TEXTS (first 3)

Sample 1 (length: 1284 chars):
Předseda PSP Milan Uhde Vážené paní poslankyně  vážení páni poslanci  vážené dámy a pánové  zahajuji slavnostní schůzi Poslanecké sněmovny Parlamentu České republiky a všechny vás vítám  Zvlášť srdečně vítám členy vlády České republiky  diplomatický sbor  představitele politického života  mezi nimi poslance bývalého Federálního shromáždění  a všechny čestné hosty   Scházíme se poprvé jako Poslanecká sněmovna Parlamentu České republiky  a to v prvý den její samostatnosti  Proto má dnešní schůze s
...

Sample 2 (length: 175 chars):
Místopředseda PSP Jiří Vlach Vážené dámy a pánové  přistoupíme k prvému bodu pořadu naší schůze  a prosím  aby se slova ujal předseda Poslanecké sněmovny pan Milan Uhde       


Samp

In [15]:
df_A = pd.read_csv('/home/tom/data/italy/camera_2006-2022.csv')
df_B = pd.read_csv('/home/tom/data/italy/camera_1992-2006.csv')
concat = [df_A, df_B]
df = pd.concat(concat)
df.to_csv('/home/tom/data/italy/italy.csv')

/tmp/ipykernel_679626/220204005.py:2: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_B = pd.read_csv('/home/tom/data/italy/camera_1992-2006.csv')
